# Notebook B — SemSeg + transfer learning for environment classification

**Task:** same multi-label environment classification as Notebook A
(`vegetation, water, city`), but via **semantic segmentation + transfer
learning**: fine-tune a pretrained SegFormer head on the 3 environment classes (+ background), then derive the
per-frame multi-label by thresholding each class's **pixel-area fraction**.

Outputs per-frame predictions to `dataset/eval/env_pred_semseg.csv` and runtime/frame to
`dataset/eval/runtime_semseg.json`, in the same format as Notebook A for `seg_evaluation.ipynb`.


> **Tuning note (2026-07-09):** training data composition, LR and the
> per-class presence thresholds below were tuned on a dedicated dev set
> (`dataset/eval/dev_mapillary.csv`, disjoint from training AND from the
> hand-labeled test set). Full experiment log: `research/seg_finetune_tuning.md`.
> Key findings: Vistas alone cannot teach `water` (24 usable images -> F1 0.0);
> class-balanced sampling + water oversampling + 258 ADE20K water scenes fix the
> segmentation (water IoU 0.0 -> 0.77, mIoU 0.64 -> 0.85).


## 0. Dependencies
Fine-tuning needs `datasets` + `accelerate` (run once if missing).

In [1]:
# !pip install accelerate
# (ADE20K now loads from the official CSAIL zip, so the `datasets` package is NOT required.)
print("If the Trainer import fails, uncomment the pip line above and re-run.")

If the Trainer import fails, uncomment the pip line above and re-run.


## 1. Setup

In [ ]:
import sys, json, time
from pathlib import Path

import numpy as np
import cv2
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
import segmentation_common as sc

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
ENV_CLASSES = sc.CATEGORIES["environment"]
ENV_ID = {c: i for i, c in enumerate(ENV_CLASSES)}   # env-only label space 0..3

# Explicit background class: map all non-env pixels to 'other' (id 4) instead of
# ignoring them, so the model can output "none of the 4" at inference. This
# markedly cleaned up the env area-fractions on own frames (label-acc 0.65->0.86).
USE_OTHER = True
OTHER_ID = len(ENV_CLASSES)          # == 4
N_OUT = len(ENV_CLASSES) + (1 if USE_OTHER else 0)
print("Device:", DEVICE, "| environment classes:", ENV_CLASSES)
if DEVICE == "cpu":
    print("WARNING: training on CPU is slow - keep MAX_STEPS small for a smoke run.")

## 2. Configuration

In [ ]:
BASE_MODEL = "nvidia/segformer-b0-finetuned-ade-512-512"  # ADE encoder has a water prior (tuning opt 1)
DATASET = "MAPILLARY"                     # "MAPILLARY" (POV-matched, street-level) | "ADE20K"
MAPILLARY_ROOT = Path("../dataset/external/mapillary_vistas")
OUTPUT_DIR = Path("../models/segformer_env")
SEED = 42

# --- Test set: the hand-labeled eval data. NEVER train on these images. ---
# build_test_dataset.py drew some Mapillary/ADE20K images INTO this test set, so
# candidates.csv is read below and those exact images are held out of training.
TEST_IMAGES = Path("../dataset/test_images")
CANDIDATES_CSV = TEST_IMAGES / "candidates.csv"
TEST_LABELS_CSV = TEST_IMAGES / "labels.csv"

# --- Training config -----------------------------------------------------------
# Tuned on the dedicated dev set dataset/eval/dev_mapillary.csv (disjoint from
# training and test); full log: research/seg_finetune_tuning.md
NUM_EPOCHS = 3
BATCH_SIZE = 4
LR = 2e-4                    # E2/E6: fresh decode head needs more than the 6e-5 default
TRAIN_LIMIT = 4000           # size of the balanced subset; None = all ~17.8k
VAL_LIMIT = 200              # in-training mIoU monitor slice
MAX_STEPS = None             # set e.g. 50 for a quick smoke run
BALANCED_SAMPLING = True     # E3/E4: class-balanced sample via the share cache
OVERSAMPLE_WATER = 2         # E4/E6: repeat the ~24 water images (Vistas water scarcity)
ADE_WATER_AUG = 258          # E5/E6: add ADE20K water scenes (test-set stems excluded)

# Per-class presence thresholds (pixel-area fraction), tuned on the dev set.
# A single global 3% is wrong per class: city needs a much higher bar (buildings
# appear somewhere in nearly every street image), water even more so.
AREA_THRESHOLDS = {"vegetation": 0.25, "water": 0.01, "city": 0.01}  # 3-class, val-tuned

PRED_CSV = Path("../dataset/eval/env_pred_semseg.csv")
RUNTIME_JSON = Path("../dataset/eval/runtime_semseg.json")


## 3. Label harmonization -> environment-only label space

Source class names are remapped to the 5 environment ids; everything else -> VOID (ignored).

In [ ]:
def ade20k_source_names():
    from huggingface_hub import hf_hub_download
    import json as _json
    p = hf_hub_download("huggingface/label-files", "ade20k-id2label.json", repo_type="dataset")
    id2label = _json.load(open(p))
    n = max(int(k) for k in id2label) + 1
    names = ["other"] * (n + 1)          # ADE masks are 1-indexed; 0 == other
    for k, v in id2label.items():
        names[int(k) + 1] = v
    return names


# ADE20K LUT is always built: it is the fallback branch AND the water-augmentation
# source for the Mapillary branch.
LUT_ADE = sc.build_id_lookup(ade20k_source_names(), sc.ADE20K_TO_TAXONOMY, class_id=ENV_ID)

if DATASET == "ADE20K":
    LUT = LUT_ADE
elif DATASET == "MAPILLARY":
    import json as _json
    cfg = _json.load(open(MAPILLARY_ROOT / "config_v2.0.json"))
    LUT = sc.build_id_lookup([lbl["name"] for lbl in cfg["labels"]],
                             sc.MAPILLARY_TO_TAXONOMY, class_id=ENV_ID)
else:
    raise ValueError(DATASET)

if USE_OTHER:  # background (unmapped -> VOID) becomes an explicit 'other' class
    LUT = np.where(LUT == sc.VOID_ID, OTHER_ID, LUT)
if 'LUT_ADE' in dir():
    if USE_OTHER:
        LUT_ADE = np.where(LUT_ADE == sc.VOID_ID, OTHER_ID, LUT_ADE)

print(f"{DATASET}: {int((LUT != sc.VOID_ID).sum())} source classes map into the {len(ENV_CLASSES)} env classes")


## 4. Dataset & preprocessing

In [ ]:
from torch.utils.data import Dataset
from transformers import SegformerImageProcessor
from PIL import Image
import csv as _csv
import random as _random
import urllib.request, zipfile

processor = SegformerImageProcessor.from_pretrained(BASE_MODEL, do_reduce_labels=False)


class SegDataset(Dataset):
    def __init__(self, items):
        self.items = items                # list of (load_img, load_mask, lut) triples

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        load_img, load_mask, lut = self.items[i]
        mask = lut[load_mask()].astype(np.uint8)             # remap to env ids
        enc = processor(load_img(), mask, return_tensors="pt")
        return {"pixel_values": enc["pixel_values"][0], "labels": enc["labels"][0]}


# --- ADE20K: fallback branch + water-augmentation source ------------------------
ADE_URL = "http://data.csail.mit.edu/places/ADEchallenge/ADEChallengeData2016.zip"
ADE_ROOT = Path("../dataset/external/ADEChallengeData2016")


def ensure_ade20k() -> Path:
    if ADE_ROOT.exists():
        return ADE_ROOT
    ADE_ROOT.parent.mkdir(parents=True, exist_ok=True)
    zip_path = ADE_ROOT.parent / "ADEChallengeData2016.zip"
    if not zip_path.exists():
        print("Downloading ADE20K (~1 GB, one-time)...")
        urllib.request.urlretrieve(ADE_URL, zip_path)
    print("Extracting...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(ADE_ROOT.parent)
    return ADE_ROOT


def build_ade20k_items(split, limit=None):        # split: "training" | "validation"
    root = ensure_ade20k()
    img_dir, ann_dir = root / "images" / split, root / "annotations" / split
    imgs = sorted(img_dir.glob("*.jpg"))[:limit]
    return [(lambda p=p: np.array(Image.open(p).convert("RGB")),
             lambda a=ann_dir / (p.stem + ".png"): np.array(Image.open(a)),
             LUT_ADE) for p in imgs]


# water-dominated ADE20K scene categories (same set as scripts/build_test_dataset.py)
ADE_WATER_SCENES = {
    "coast", "beach", "river", "creek", "harbor", "marsh", "lagoon", "water",
    "waterway", "shore", "tidal_river", "bayou", "swamp", "pond", "millpond",
    "fishpond", "waterscape", "foreshore", "dock", "pier", "floating_dock",
}


def build_ade_water_items(cap):
    """Cross-dataset water augmentation (tuning experiment E5/E6).

    Vistas water is near-absent (~24 usable training images -> water F1 0.0
    without this); ADE20K has hundreds of masked water scenes. Test-set ADE20K
    stems from candidates.csv are excluded to stay leakage-free.
    """
    root = ensure_ade20k()
    test_stems = {Path(r["orig_path"]).stem for r in _csv.DictReader(open(CANDIDATES_CSV))
                  if r.get("source") == "ade20k"}
    stems = []
    for line in open(root / "sceneCategories.txt"):
        parts = line.split()
        if len(parts) == 2 and parts[1] in ADE_WATER_SCENES \
                and "_train_" in parts[0] and parts[0] not in test_stems:
            stems.append(parts[0])
    _random.Random(SEED).shuffle(stems)
    stems = stems[:cap]
    img_dir, ann_dir = root / "images" / "training", root / "annotations" / "training"
    return [(lambda p=img_dir / f"{s}.jpg": np.array(Image.open(p).convert("RGB")),
             lambda a=ann_dir / f"{s}.png": np.array(Image.open(a)),
             LUT_ADE) for s in stems]


# --- Mapillary Vistas ------------------------------------------------------------
def mapillary_test_stems(candidates_csv):
    """Basenames of Mapillary images already in the hand-labeled test set."""
    stems = set()
    if Path(candidates_csv).exists():
        for row in _csv.DictReader(open(candidates_csv)):
            if row.get("source") == "mapillary":
                stems.add(Path(row["orig_path"]).stem)
    return stems


def dev_reserved_stems():
    """Training-split stems reserved into the HP-tuning dev set
    (dataset/eval/dev_mapillary.csv) - never train on them either."""
    p = Path("../dataset/eval/dev_mapillary.csv")
    if not p.exists():
        return set()
    return {r["stem"] for r in _csv.DictReader(open(p))
            if r.get("split") == "training"}


def _mapillary_label_dir(split):
    for d in (MAPILLARY_ROOT / split / "v2.0" / "labels",
              MAPILLARY_ROOT / split / "labels"):
        if d.exists():
            return d
    raise FileNotFoundError(f"no Mapillary labels found for split {split!r}")


def load_vistas_mask(p):
    arr = np.array(Image.open(p))
    return arr[..., 0] if arr.ndim == 3 else arr


def build_mapillary_items(split, exclude_stems=frozenset(), stems=None, limit=None):
    img_dir = MAPILLARY_ROOT / split / "images"
    lbl_dir = _mapillary_label_dir(split)
    if stems is None:
        stems = [p.stem for p in sorted(img_dir.glob("*.jpg"))]
    items = []
    for s in stems:                     # may contain repeats (oversampling)
        if s in exclude_stems:
            continue
        img, mask = img_dir / f"{s}.jpg", lbl_dir / f"{s}.png"
        if not mask.exists():
            continue
        items.append((lambda p=img: np.array(Image.open(p).convert("RGB")),
                      lambda a=mask: load_vistas_mask(a), LUT))
        if limit and len(items) >= limit:
            break
    return items


def balanced_training_stems(limit, oversample_water, exclude):
    """Class-balanced training sample (tuning experiments E3/E4).

    Uses the per-image class-share cache built by
    `scripts/finetune_experiments.py shares`. Scarce classes get first pick of
    each image; the handful of water images is additionally repeated, because a
    random sample contains ~2 water images and the head then never learns the
    class (E1: water F1 = 0.0).
    """
    shares = Path("../dataset/eval/mapillary_train_shares.csv")
    if not shares.exists():
        print("WARNING: share cache missing -> falling back to a random sample."
              " Run: python scripts/finetune_experiments.py shares")
        return None
    rows = [r for r in _csv.DictReader(open(shares)) if r["stem"] not in exclude]
    rules = [("water", 0.01), ("vegetation", 0.20), ("city", 0.25)]
    quota = (limit or len(rows)) // len(rules)
    rng = _random.Random(SEED)
    rng.shuffle(rows)
    buckets, used = {c: [] for c, _ in rules}, set()
    for r in rows:
        for cls, thr in rules:
            if float(r[f"share_{cls}"]) >= thr and r["stem"] not in used:
                if len(buckets[cls]) < quota:
                    buckets[cls].append(r["stem"])
                    used.add(r["stem"])
                break  # first matching rule claims the image, full bucket or not
    print("balanced buckets:", {k: len(v) for k, v in buckets.items()})
    buckets["water"] = buckets["water"] * max(1, oversample_water)
    stems = [s for b in buckets.values() for s in b]
    if limit and len(stems) < limit:  # top up with random unused images
        img_dir = MAPILLARY_ROOT / "training" / "images"
        all_stems = sorted(p.stem for p in img_dir.glob("*.jpg")
                           if p.stem not in exclude)
        stems += [s for s in all_stems if s not in used][:limit - len(stems)]
    rng.shuffle(stems)
    return stems


if DATASET == "ADE20K":
    train_ds = SegDataset(build_ade20k_items("training"))
    val_ds = SegDataset(build_ade20k_items("validation"))
    print("train/val sizes:", len(train_ds), len(val_ds))
elif DATASET == "MAPILLARY":
    held_out = mapillary_test_stems(CANDIDATES_CSV) | dev_reserved_stems()
    print(f"held out of training: {len(held_out)} stems (test set + dev set)")
    stems = (balanced_training_stems(TRAIN_LIMIT, OVERSAMPLE_WATER, held_out)
             if BALANCED_SAMPLING else None)
    items = build_mapillary_items("training", held_out, stems=stems,
                                  limit=None if stems else TRAIN_LIMIT)
    if ADE_WATER_AUG:
        extra = build_ade_water_items(ADE_WATER_AUG)
        items += extra
        print(f"+ {len(extra)} ADE20K water-scene images (cross-dataset augmentation)")
    _random.Random(SEED).shuffle(items)
    train_ds = SegDataset(items)
    val_ds = SegDataset(build_mapillary_items("validation", held_out, limit=VAL_LIMIT))
    print("train/val sizes:", len(train_ds), len(val_ds))


## 5. Model (transfer learning: new 5-class head)

In [ ]:
from transformers import SegformerForSemanticSegmentation


class MPSSafeBatchNorm2d(torch.nn.BatchNorm2d):
    """Decomposed BatchNorm forward on MPS (train mode only).

    torch 2.12 on MPS: native_batch_norm's backward raises
    'view size is not compatible ...' - the single BN in SegFormer's decode
    head makes training crash. Decomposing into mean/var primitives sidesteps
    the bug; numerics and running-stat updates match nn.BatchNorm2d exactly.
    """

    def forward(self, x):
        if not self.training or x.device.type != "mps":
            return super().forward(x)
        mean = x.mean(dim=(0, 2, 3))
        var = x.var(dim=(0, 2, 3), unbiased=False)
        if self.track_running_stats:
            with torch.no_grad():
                n = x.numel() / x.shape[1]
                self.running_mean.lerp_(mean, self.momentum)
                self.running_var.lerp_(var * n / (n - 1), self.momentum)
                self.num_batches_tracked += 1
        xhat = (x - mean[None, :, None, None]) / torch.sqrt(var[None, :, None, None] + self.eps)
        return xhat * self.weight[None, :, None, None] + self.bias[None, :, None, None]


id2label = {i: c for c, i in ENV_ID.items()}
label2id = dict(ENV_ID)
if USE_OTHER:
    id2label[OTHER_ID] = "other"; label2id["other"] = OTHER_ID
model = SegformerForSemanticSegmentation.from_pretrained(
    BASE_MODEL,
    num_labels=N_OUT,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,        # replace the Cityscapes head with a fresh env-class one
)
_bn = model.decode_head.batch_norm
_safe = MPSSafeBatchNorm2d(_bn.num_features, eps=_bn.eps, momentum=_bn.momentum)
_safe.load_state_dict(_bn.state_dict())
model.decode_head.batch_norm = _safe
model = model.to(DEVICE)
print("env loss ignore_index:", model.config.semantic_loss_ignore_index, "(== VOID 255)")


## 6. Train

In [ ]:
from transformers import TrainingArguments, Trainer


class SegTrainer(Trainer):
    """Cross-entropy at logits resolution (labels nearest-downsampled 4x).

    Avoids the in-model 4x logits upsample: faster, and its bilinear backward
    was a second MPS-crash suspect. Supervision semantics are unchanged -
    SegFormer's logits are stride-4 anyway.
    """

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs["labels"]
        outputs = model(pixel_values=inputs["pixel_values"])
        logits = outputs.logits
        small = F.interpolate(labels[:, None].float(), size=logits.shape[-2:],
                              mode="nearest")[:, 0].long()
        loss = F.cross_entropy(logits, small, ignore_index=sc.VOID_ID)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = torch.tensor(logits)
    up = F.interpolate(logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
    preds = up.argmax(1).numpy()
    cm = np.zeros((len(ENV_CLASSES), len(ENV_CLASSES)), np.int64)
    for p, g in zip(preds, labels):
        valid = g != sc.VOID_ID
        idx = g[valid] * len(ENV_CLASSES) + p[valid]
        cm += np.bincount(idx, minlength=len(ENV_CLASSES) ** 2).reshape(cm.shape)
    tp = np.diag(cm); denom = cm.sum(0) + cm.sum(1) - tp
    iou = np.where(denom > 0, tp / np.clip(denom, 1, None), np.nan)
    return {"mIoU": float(np.nanmean(iou))}


args = TrainingArguments(
    output_dir=str(OUTPUT_DIR), learning_rate=LR, num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS if MAX_STEPS else -1,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="mIoU", greater_is_better=True, logging_steps=20,
    remove_unused_columns=False, seed=SEED,
)
trainer = SegTrainer(model=model, args=args, train_dataset=train_ds,
                     eval_dataset=val_ds, compute_metrics=compute_metrics)
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
processor.save_pretrained(str(OUTPUT_DIR))
print("Saved fine-tuned env model to", OUTPUT_DIR)


## 7. SemSeg -> multi-label classifier (area threshold)

In [ ]:
@torch.no_grad()
def segment_env(image_rgb: np.ndarray) -> np.ndarray:
    enc = processor(image_rgb, return_tensors="pt").to(DEVICE)
    logits = model(**enc).logits
    up = F.interpolate(logits, size=image_rgb.shape[:2], mode="bilinear", align_corners=False)
    return up.argmax(1)[0].to(torch.uint8).cpu().numpy()


def classify_environment_semseg(image_rgb: np.ndarray, thresholds: dict = None) -> dict:
    """Multi-label prediction: class present if its pixel-area fraction exceeds
    the per-class threshold (tuned on the dev set - see AREA_THRESHOLDS)."""
    thresholds = thresholds or AREA_THRESHOLDS
    mask = segment_env(image_rgb)
    n = mask.size
    return {c: int((mask == ENV_ID[c]).sum() / n > thresholds[c]) for c in ENV_CLASSES}


## 8. Predict over the test set + runtime

In [ ]:
def run_semseg_testset() -> pd.DataFrame:
    # Recurse the labeled test set (ade20k/, mapillary/, own_frames/); the
    # 'filename' column matches labels.csv so seg_evaluation.ipynb can join them.
    exts = {".jpg", ".jpeg", ".png"}
    imgs = sorted(p for p in TEST_IMAGES.rglob("*")
                  if p.suffix.lower() in exts and "overview" not in p.parts)
    if not imgs:
        print("No test images found - run scripts/build_test_dataset.py first.")
        return pd.DataFrame()

    rows, t0 = [], time.perf_counter()
    for fp in imgs:
        img = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
        rel = str(fp.relative_to(TEST_IMAGES))
        rows.append({"filename": rel, **classify_environment_semseg(img)})
    ms = (time.perf_counter() - t0) / len(imgs) * 1000

    df = pd.DataFrame(rows)
    PRED_CSV.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(PRED_CSV, index=False)
    json.dump({"method": "semseg", "ms_per_frame": ms, "n": len(imgs)}, open(RUNTIME_JSON, "w"))
    print(f"Saved {len(df)} predictions -> {PRED_CSV}  |  {ms:.1f} ms/frame")
    return df


predictions = run_semseg_testset()
predictions.head()